<a href="https://colab.research.google.com/github/rzangef21/spb-kelompok-2/blob/main/Certainty_Factor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

df = pd.read_csv('/Dataset_indomaret_sales.csv')
df.head()

,Transaction_ID,Date,Product_Name,Category,Units_Sold,Unit_Price,Total_Revenue,Store_Location,Payment_Method
0,T0001,2024-10-06,Pepsodent Toothpaste,Personal Care,NaN,15000,675000.0,Jakarta,Cash
1,T0002,2024-10-01,ABC Kecap Manis 620ml,Groceries,60.0,18000,1080000.0,Jakarta,Card
2,T0003,2024-10-01,Lifebuoy Body Wash,Personal Care,17.0,25000,425000.0,Medan,Cash
3,T0004,2024-10-06,Milo 1kg,Drinks,95.0,90000,8550000.0,Bandung,Cash
4,T0005,2024-10-03,Indomie Goreng,Instant Noodles,30.0,3000,90000.0,Surabaya,Card


In [23]:
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')

In [24]:
df['Units_Sold'] = df['Units_Sold'].fillna(df['Units_Sold'].median())
df['Unit_Price'] = df['Unit_Price'].fillna(df['Unit_Price'].median())

In [6]:
df.loc[df['Total_Revenue'].isna(), 'Total_Revenue'] = \
    df['Units_Sold'] * df['Unit_Price']

In [25]:
df['Date'] = pd.to_datetime(df['Date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Transaction_ID  4000 non-null   object        
 1   Date            4000 non-null   datetime64[ns]
 2   Product_Name    4000 non-null   object        
 3   Category        4000 non-null   object        
 4   Units_Sold      4000 non-null   float64       
 5   Unit_Price      4000 non-null   float64       
 6   Total_Revenue   4000 non-null   float64       
 7   Store_Location  4000 non-null   object        
 8   Payment_Method  4000 non-null   object        
 9   Year            4000 non-null   int32         
 10  Month           4000 non-null   int32         
 11  Day             4000 non-null   int32         
 12  Sales_Level     4000 non-null   category      
dtypes: category(1), datetime64[ns](1), float64(3), int32(3), object(5)
memory usage: 332.3+ KB


In [8]:
df['Transaction_ID'].duplicated().sum()

np.int64(0)

In [9]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

In [10]:
df['Sales_Level'] = pd.qcut(df['Total_Revenue'], q=3, labels=['Low','Medium','High'])

In [11]:
store_perf = df.groupby('Store_Location')['Total_Revenue'].sum().reset_index()

In [12]:
best_store = store_perf.loc[store_perf['Total_Revenue'].idxmax()]
worst_store = store_perf.loc[store_perf['Total_Revenue'].idxmin()]

In [13]:
avg = store_perf['Total_Revenue'].mean()

store_perf['Status'] = store_perf['Total_Revenue'].apply(
    lambda x: 'Unggul' if x > avg else ('Rendah' if x < avg else 'Rata-rata')
)

In [14]:
store_perf

,Store_Location,Total_Revenue,Status
0,Bandung,1.091441e+09,Rendah
1,Jakarta,1.326678e+09,Unggul
2,Medan,1.106365e+09,Rendah
3,Surabaya,1.150999e+09,Rendah


In [27]:
#CF
df_mb = pd.DataFrame(list(mb_values.items()), columns=['Category', 'Measure of Belief (MB)'])

df_mb = df_mb.sort_values(by='Measure of Belief (MB)', ascending=False).reset_index(drop=True)

df_mb['CF_Pakar'] = cf_pakar

df_mb['Estimated_CF'] = df_mb['Measure of Belief (MB)'] * df_mb['CF_Pakar']

print("Tabel Kekuatan Aturan (Rule Strength) berdasarkan Certainty Factor:")
display(df_mb)

Tabel Kekuatan Aturan (Rule Strength) berdasarkan Certainty Factor:


,Category,Measure of Belief (MB),CF_Pakar,Estimated_CF
0,Smokes,0.741294,0.8,0.593035
1,Personal Care,0.572146,0.8,0.457716
2,Groceries,0.524631,0.8,0.419704
3,Drinks,0.326923,0.8,0.261538
4,Snacks,0.075410,0.8,0.060328
5,Health,0.000000,0.8,0.000000
6,Instant Noodles,0.000000,0.8,0.000000


In [28]:
md = 0.1
evidence_user = 1.0

df_final_cf = pd.DataFrame(list(mb_values.items()), columns=['Category', 'MB'])

df_final_cf['MD'] = md
df_final_cf['CF_Kriteria'] = df_final_cf['MB'] - df_final_cf['MD']
df_final_cf['CF_Akhir'] = df_final_cf['CF_Kriteria'] * evidence_user * cf_pakar

df_final_cf = df_final_cf.sort_values(by='CF_Akhir', ascending=False).reset_index(drop=True)

print(f"Hasil Perhitungan Certainty Factor untuk Semua Kategori (Evidence: {evidence_user}):")
display(df_final_cf)

Hasil Perhitungan Certainty Factor untuk Semua Kategori (Evidence: 1.0):


,Category,MB,MD,CF_Kriteria,CF_Akhir
0,Smokes,0.741294,0.1,0.641294,0.513035
1,Personal Care,0.572146,0.1,0.472146,0.377716
2,Groceries,0.524631,0.1,0.424631,0.339704
3,Drinks,0.326923,0.1,0.226923,0.181538
4,Snacks,0.075410,0.1,-0.024590,-0.019672
5,Health,0.000000,0.1,-0.100000,-0.080000
6,Instant Noodles,0.000000,0.1,-0.100000,-0.080000
